In [ ]:
# Import libraries
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import sys
sys.path.append('..')

# Import custom FastF1 loader
from src.fastf1_loader import (
    get_session, get_race_results, get_lap_data,
    get_telemetry, get_weather_data, get_schedule,
    compare_drivers_lap_times
)

# Enable FastF1 cache
fastf1.Cache.enable_cache('../cache')

plt.style.use('seaborn-v0_8-darkgrid')
print('FastF1 loaded successfully!')
print(f'FastF1 version: {fastf1.__version__}')

## 1. Get 2025 Season Schedule

FastF1 menyediakan akses ke data telemetri F1 resmi. Di sini kita load kalender 2025.

In [ ]:
# Get 2025 schedule
schedule = get_schedule(2025)
print('2025 F1 Calendar:')
schedule[['RoundNumber', 'EventName', 'Country', 'Location', 'EventDate']]

## 2. Load Race Session Data

In [ ]:
# Load a race session (example: Australia 2025 - first race)
session = get_session(2025, 'Australia', 'R')
print(f'Session loaded: {session.event["EventName"]} - {session.name}')

In [ ]:
# Get race results
results = get_race_results(session)
print('Race Results:')
results[['Position', 'Abbreviation', 'TeamName', 'Time', 'Status', 'Points']]

## 3. Lap Time Analysis

In [ ]:
# Get lap data
laps = get_lap_data(session)
print(f'Total laps: {len(laps)}')
laps.head()

In [ ]:
# Convert lap times to seconds for easier analysis
laps['LapTime_sec'] = laps['LapTime'].dt.total_seconds()

# Filter valid laps (remove outliers like pit laps, safety car, etc.)
valid_laps = laps[(laps['LapTime_sec'] > 80) & (laps['LapTime_sec'] < 120)]

# Average lap time per driver
avg_lap_times = valid_laps.groupby('Driver')['LapTime_sec'].mean().sort_values()

fig = px.bar(avg_lap_times.reset_index(name='Avg Lap Time (sec)'),
             x='Avg Lap Time (sec)', y='Driver', orientation='h',
             color='Avg Lap Time (sec)', color_continuous_scale='RdYlGn_r',
             title='Average Lap Time per Driver')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_dark')
fig.show()

In [ ]:
# Lap time evolution
top_3_drivers = results.head(3)['Abbreviation'].tolist()
top_3_laps = valid_laps[valid_laps['Driver'].isin(top_3_drivers)]

fig = px.line(top_3_laps, x='LapNumber', y='LapTime_sec', color='Driver',
              title='Lap Time Evolution - Top 3 Finishers')
fig.update_layout(template='plotly_dark', xaxis_title='Lap', yaxis_title='Lap Time (sec)')
fig.show()

## 4. Fastest Laps Comparison

In [ ]:
# Get fastest lap for each driver
fastest_laps = laps.loc[laps.groupby('Driver')['LapTime_sec'].idxmin()]
fastest_laps = fastest_laps.sort_values('LapTime_sec')[['Driver', 'Team', 'LapNumber', 'LapTime_sec', 'Compound']]

print('Fastest Lap per Driver:')
fastest_laps

In [ ]:
# Gap to fastest lap
fastest_overall = fastest_laps['LapTime_sec'].min()
fastest_laps['Gap_to_Fastest'] = fastest_laps['LapTime_sec'] - fastest_overall

fig = px.bar(fastest_laps, x='Gap_to_Fastest', y='Driver', orientation='h',
             color='Team', title='Gap to Fastest Lap')
fig.update_layout(yaxis={'categoryorder':'total ascending'}, template='plotly_dark')
fig.show()

## 5. Tyre Strategy Analysis

In [ ]:
# Compound usage per driver
compound_usage = laps.groupby(['Driver', 'Compound']).size().unstack(fill_value=0)

fig = px.imshow(compound_usage, 
                title='Tyre Compound Usage (Laps per Compound)',
                labels={'color': 'Laps'},
                color_continuous_scale='YlOrRd')
fig.update_layout(template='plotly_dark')
fig.show()

In [ ]:
# Stint length by compound
stint_data = laps.groupby(['Driver', 'Stint'])['Compound'].first().reset_index()
stint_lengths = laps.groupby(['Driver', 'Stint']).size().reset_index(name='StintLength')
stint_data = stint_data.merge(stint_lengths, on=['Driver', 'Stint'])

fig = px.box(stint_data, x='Compound', y='StintLength', color='Compound',
             title='Stint Length by Tyre Compound')
fig.update_layout(template='plotly_dark')
fig.show()

## 6. Telemetry Analysis (Speed, Throttle, Brake)

In [ ]:
# Get telemetry for fastest lap
winner = results.iloc[0]['Abbreviation']
winner_fastest = laps.pick_driver(winner).pick_fastest()

telemetry = get_telemetry(winner_fastest)
print(f'Telemetry data for {winner} fastest lap:')
print(f'Data points: {len(telemetry)}')
telemetry.head()

In [ ]:
# Speed trace
fig = go.Figure()
fig.add_trace(go.Scatter(x=telemetry['Distance'], y=telemetry['Speed'],
                         mode='lines', name='Speed'))
fig.update_layout(title=f'{winner} Speed Trace - Fastest Lap',
                  xaxis_title='Distance (m)', yaxis_title='Speed (km/h)',
                  template='plotly_dark')
fig.show()

In [ ]:
# Combined telemetry plot
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    subplot_titles=('Speed', 'Throttle', 'Brake'),
                    vertical_spacing=0.08)

fig.add_trace(go.Scatter(x=telemetry['Distance'], y=telemetry['Speed'],
                         line=dict(color='cyan'), name='Speed'), row=1, col=1)
fig.add_trace(go.Scatter(x=telemetry['Distance'], y=telemetry['Throttle'],
                         line=dict(color='green'), name='Throttle'), row=2, col=1)
fig.add_trace(go.Scatter(x=telemetry['Distance'], y=telemetry['Brake'].astype(int) * 100,
                         line=dict(color='red'), name='Brake'), row=3, col=1)

fig.update_layout(height=700, title_text=f'{winner} Telemetry - Fastest Lap',
                  template='plotly_dark', showlegend=False)
fig.update_xaxes(title_text='Distance (m)', row=3, col=1)
fig.show()

## 7. Driver Comparison

In [ ]:
# Compare two drivers lap times
driver1 = results.iloc[0]['Abbreviation']
driver2 = results.iloc[1]['Abbreviation']

# Get lap times for both drivers
d1_laps = laps.pick_driver(driver1)[['LapNumber', 'LapTime']].copy()
d2_laps = laps.pick_driver(driver2)[['LapNumber', 'LapTime']].copy()

d1_laps['LapTime_sec'] = d1_laps['LapTime'].dt.total_seconds()
d2_laps['LapTime_sec'] = d2_laps['LapTime'].dt.total_seconds()

fig = go.Figure()
fig.add_trace(go.Scatter(x=d1_laps['LapNumber'], y=d1_laps['LapTime_sec'],
                         mode='lines+markers', name=driver1))
fig.add_trace(go.Scatter(x=d2_laps['LapNumber'], y=d2_laps['LapTime_sec'],
                         mode='lines+markers', name=driver2))
fig.update_layout(title=f'{driver1} vs {driver2} Lap Times - Australia 2025',
                  xaxis_title='Lap', yaxis_title='Lap Time (sec)',
                  template='plotly_dark')
fig.show()

In [ ]:
# Speed comparison on fastest laps
d1_fastest = laps.pick_driver(driver1).pick_fastest()
d2_fastest = laps.pick_driver(driver2).pick_fastest()

d1_tel = get_telemetry(d1_fastest)
d2_tel = get_telemetry(d2_fastest)

fig = go.Figure()
fig.add_trace(go.Scatter(x=d1_tel['Distance'], y=d1_tel['Speed'],
                         mode='lines', name=driver1))
fig.add_trace(go.Scatter(x=d2_tel['Distance'], y=d2_tel['Speed'],
                         mode='lines', name=driver2))
fig.update_layout(title=f'Speed Comparison: {driver1} vs {driver2} - Australia 2025',
                  xaxis_title='Distance (m)', yaxis_title='Speed (km/h)',
                  template='plotly_dark')
fig.show()

## 8. Weather Analysis

In [ ]:
# Get weather data
weather = get_weather_data(session)
print('Weather Data:')
weather.head(10)

In [ ]:
# Temperature evolution during race
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Air Temperature', 'Track Temperature'),
                    vertical_spacing=0.1)

fig.add_trace(go.Scatter(y=weather['AirTemp'], mode='lines',
                         line=dict(color='skyblue'), name='Air Temp'), row=1, col=1)
fig.add_trace(go.Scatter(y=weather['TrackTemp'], mode='lines',
                         line=dict(color='orange'), name='Track Temp'), row=2, col=1)

fig.update_layout(height=500, title_text='Temperature During Race',
                  template='plotly_dark', showlegend=False)
fig.update_yaxes(title_text='Air Temp (C)', row=1, col=1)
fig.update_yaxes(title_text='Track Temp (C)', row=2, col=1)
fig.show()

## 9. Sector Times Analysis

In [ ]:
# Sector times for top 10
top_10 = results.head(10)['Abbreviation'].tolist()
top_10_laps = laps[laps['Driver'].isin(top_10)]

# Convert sector times
for s in ['Sector1Time', 'Sector2Time', 'Sector3Time']:
    top_10_laps[f'{s}_sec'] = top_10_laps[s].dt.total_seconds()

# Best sector times per driver
best_sectors = top_10_laps.groupby('Driver').agg({
    'Sector1Time_sec': 'min',
    'Sector2Time_sec': 'min',
    'Sector3Time_sec': 'min'
}).round(3)

print('Best Sector Times (Top 10 Finishers):')
best_sectors

In [ ]:
# Sector time heatmap
fig = px.imshow(best_sectors.T, 
                title='Best Sector Times Heatmap',
                labels={'color': 'Time (sec)'},
                color_continuous_scale='RdYlGn_r',
                aspect='auto')
fig.update_layout(template='plotly_dark')
fig.show()

## 10. Position Changes Animation

In [ ]:
# Position per lap for all drivers
position_data = laps[['Driver', 'LapNumber', 'Position']].dropna()

# Create animated bar chart race
fig = px.bar(position_data, x='Position', y='Driver', 
             animation_frame='LapNumber',
             orientation='h',
             title='Position Changes Throughout Race',
             range_x=[0, 21])
fig.update_layout(template='plotly_dark', 
                  yaxis={'categoryorder':'total ascending'},
                  height=600)
fig.show()

---
## Summary

Notebook ini menggunakan FastF1 untuk analisis lanjutan:
- Session dan race results loading
- Lap time analysis dan comparison
- Telemetry data (speed, throttle, brake)
- Tyre strategy analysis
- Weather data
- Sector times comparison
- Driver head-to-head comparison

FastF1 menyediakan data resmi F1 yang lebih detail dibanding CSV dataset.